## Adaptive Recommendation
Use the existing runtime and learner store for a complete recommendation-and-feedback example.

In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import os

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/hlo_recommender").is_dir())
os.chdir(ROOT)

from hlo_recommender.cli import parser, run
from hlo_recommender.io import read_jsonl, write_json, write_jsonl

SESSION_FILE = ROOT / "artifacts/notebooks/session.json"
SESSION = json.loads(SESSION_FILE.read_text()) if SESSION_FILE.exists() else {}


In [ ]:
from datetime import datetime, timedelta, timezone
from hlo_recommender.pipeline import Runtime
from hlo_recommender.models.bkt import LearnerStore, load_parameters
from hlo_recommender.knowledge import HLO

runtime = Runtime.from_file(ROOT / "configs/default.toml")
OUTPUT = ROOT / "artifacts/notebooks" / f"recommendation-{uuid4().hex[:8]}"
OUTPUT.mkdir(parents=True)
MODEL = Path(SESSION.get("bkt_model", ROOT / "examples/results/bkt_model.json"))
parameters = load_parameters(MODEL)
QUESTION = "Find resources for arterial blood gas interpretation."
CONTEXT = {"level": "Junior", "minutes": 15}
LEARNER = "notebook-learner"
GENERATOR = "template"
generator = runtime.generator(GENERATOR, SESSION.get("mt5_checkpoint"))


### Initial Recommendation
Record an initial assessment and retrieve recommendations from the same ontology and catalog.

In [ ]:
start = datetime(2025, 1, 1, tzinfo=timezone.utc)
events = [{"event_id": "attempt-0", "learner_id": LEARNER, "skill_id": str(HLO.K002), "correct": False, "timestamp": start.isoformat(), "data_status": "synthetic"}]
store = LearnerStore(OUTPUT / "learner.sqlite", parameters)
try:
    store.observe(events[0])
    before_state = store.snapshot(LEARNER)
    before = runtime.recommend(QUESTION, CONTEXT, generator, state=before_state)
finally:
    store.close()
before["ranked"]


### Feedback
Apply scored responses through the existing BKT update, then request recommendations again.

In [ ]:
store = LearnerStore(OUTPUT / "learner.sqlite", parameters)
try:
    for i in range(1, 9):
        event = {"event_id": f"attempt-{i}", "learner_id": LEARNER, "skill_id": str(HLO.K002), "correct": True, "timestamp": (start + timedelta(minutes=i)).isoformat(), "data_status": "synthetic"}
        store.observe(event)
        events.append(event)
    after_state = store.snapshot(LEARNER)
    after = runtime.recommend(QUESTION, CONTEXT, generator, state=after_state)
finally:
    store.close()
{"before_mastery": before_state, "after_mastery": after_state, "before_top": before["ranked"][0]["title"], "after_top": after["ranked"][0]["title"], "gaps": after["gaps"]}


### Export
Save the actual recommendations, assessment events and learner states for this run.

In [ ]:
write_json(OUTPUT / "recommendations.json", {"data_status": "synthetic", "generator": GENERATOR, "question": QUESTION, "context": CONTEXT, "before": before, "after": after, "before_state": before_state, "after_state": after_state})
write_jsonl(OUTPUT / "assessments.jsonl", events)
SESSION["recommendation_directory"] = str(OUTPUT)
write_json(SESSION_FILE, SESSION)
sorted(path.name for path in OUTPUT.iterdir())
